# Using Sherlock out-of-the-box
This notebook shows how to predict a semantic type for a given table column.
The steps are basically:
- Download files for word embedding and paragraph vector feature extraction (downloads only once) and initialize feature extraction models.
- Extract features from table columns.
- Initialize Sherlock.
- Make a prediction for the feature representation of the column.

In [2]:
import numpy as np
import pandas as pd
import pyarrow as pa

from sherlock import helpers
from sherlock.deploy.model import SherlockModel
from sherlock.functional import extract_features_to_csv
from sherlock.features.paragraph_vectors import initialise_pretrained_model, initialise_nltk
from sherlock.features.preprocessing import (
    extract_features,
    convert_string_lists_to_lists,
    prepare_feature_extraction,
    load_parquet_values,
)
from sherlock.features.word_embeddings import initialise_word_embeddings

W0704 16:59:01.956686 140598070609152 transport.py:43] unable to import 'smart_open.gcs', disabling that module


In [3]:
%env PYTHONHASHSEED

UsageError: Environment does not have key: PYTHONHASHSEED


## Initialize feature extraction models

In [4]:
prepare_feature_extraction()
initialise_word_embeddings()
initialise_pretrained_model(400)
initialise_nltk()

Preparing feature extraction by downloading 4 files:
        
 ../sherlock/features/glove.6B.50d.txt, 
 ../sherlock/features/par_vec_trained_400.pkl.docvecs.vectors_docs.npy,
        
 ../sherlock/features/par_vec_trained_400.pkl.trainables.syn1neg.npy, and 
 ../sherlock/features/par_vec_trained_400.pkl.wv.vectors.npy.
        
All files for extracting word and paragraph embeddings are present.
Initialising word embeddings
Initialise Word Embeddings process took 0:00:04.039482 seconds.
Initialise Doc2Vec Model, 400 dim, process took 0:00:01.702558 seconds. (filename = ../sherlock/features/par_vec_trained_400.pkl)
Initialised NLTK, process took 0:00:00.114449 seconds.


[nltk_data] Downloading package punkt to /home/omadbek/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/omadbek/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Extract features

In [98]:
data = pd.Series(
    [
        ["Yes", "No", "True", "False"],
        ["Machine Learning", "Deep Learning", "Natural Language Processing"],
        ["Switzerland", "Italy", "Germany", "UK", "USA", "Canada"],
        ["2019", "2020", "2024", "2025", "2018"],
        ["Technical University", "Charite University", "Humbolt Univeristy", "Freie University", "Westminster International University"],
        ["This text should is only for test purposes", "The text here is an example of texts", "I don't know but it should be"]
    ],
    name="values"
)

In [99]:
data

0                               [Yes, No, True, False]
1    [Machine Learning, Deep Learning, Natural Lang...
2       [Switzerland, Italy, Germany, UK, USA, Canada]
3                       [2019, 2020, 2024, 2025, 2018]
4    [Technical University, Charite University, Hum...
5    [This text should is only for test purposes, T...
Name: values, dtype: object

In [100]:
extract_features(
    "../temporary.csv",
    data
)
feature_vectors = pd.read_csv("../temporary.csv", dtype=np.float32)

Extracting Features: 100%|██████████████████████████████████████████████████| 6/6 [00:00<00:00, 190.26it/s]

Exporting 1588 column features


In [101]:
feature_vectors

,n_[0]-agg-any,n_[0]-agg-all,n_[0]-agg-mean,n_[0]-agg-var,n_[0]-agg-min,n_[0]-agg-max,n_[0]-agg-median,n_[0]-agg-sum,n_[0]-agg-kurtosis,n_[0]-agg-skewness,...,par_vec_390,par_vec_391,par_vec_392,par_vec_393,par_vec_394,par_vec_395,par_vec_396,par_vec_397,par_vec_398,par_vec_399
0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,-3.00,0.0,...,-0.112776,0.024357,-0.068226,-0.002175,-0.080010,-0.001344,0.028820,-0.038181,0.010487,-0.092919
1,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,-3.00,0.0,...,-0.089472,0.022572,-0.030530,0.071085,-0.170950,-0.039180,-0.043484,-0.088269,-0.042446,-0.119320
2,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,-3.00,0.0,...,-0.089710,-0.043994,-0.129129,0.127777,-0.122688,-0.044351,0.023247,-0.031682,-0.026185,-0.129652
3,1.0,1.0,1.2,0.16,1.0,2.0,1.0,6.0,0.25,1.5,...,-0.084313,-0.022032,0.007015,0.016228,-0.100103,-0.039576,-0.014171,-0.129388,-0.070808,-0.130564
4,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,-3.00,0.0,...,-0.286474,0.092132,-0.048183,-0.001602,-0.181180,-0.012749,0.025904,0.144696,0.022780,-0.202460
5,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,-3.00,0.0,...,-0.170787,-0.014235,-0.006447,0.109501,-0.053329,-0.105409,-0.002421,-0.150212,-0.009998,-0.068958


In [102]:
class_names = ['age', 'case_status', 'contact_setting', 'date', 'gender', 'id', 'location',
 'medical_boolean', 'occupation', 'outcome', 'symptoms']

## Initialize Sherlock

In [103]:
model = SherlockModel(class_names, 0.35);
model.initialize_model_from_json(with_weights=True, model_id="sherlock_fine_tuned");

In [104]:
#model = finetune_model.load_weights("my_custom_sherlock_head.h5")

## Predict semantic type for column

In [105]:
predicted_labels = model.predict(feature_vectors, "sherlock_fine_tuned")

In [106]:
scores = model.predict_with_confidences(feature_vectors, top_k=3)

In [107]:
predicted_labels

array(['medical_boolean', 'unknown', 'location', 'date', 'location',
       'unknown'], dtype=object)

In [108]:
scores

[[('medical_boolean', 0.9679866433143616),
  ('case_status', 0.007304450497031212),
  ('occupation', 0.005457734689116478)],
 [('medical_boolean', 0.3141578733921051),
  ('location', 0.2686574161052704),
  ('contact_setting', 0.23042947053909302)],
 [('location', 0.3945768475532532),
  ('contact_setting', 0.1624925434589386),
  ('symptoms', 0.12103063613176346)],
 [('date', 0.9405483603477478),
  ('medical_boolean', 0.03328520432114601),
  ('age', 0.006390280555933714)],
 [('location', 0.7203510403633118),
  ('case_status', 0.11539722979068756),
  ('contact_setting', 0.04625263437628746)],
 [('location', 0.3480422794818878),
  ('medical_boolean', 0.29555743932724),
  ('outcome', 0.13080962002277374)]]